# Fraud Detection with Apache Spark

This notebook documents the reproducible Big Data pipeline for the anonymized credit-card fraud dataset. It uses Spark DataFrames, class weights for the imbalanced label, Random Forest, and fraud-class Precision/Recall/F1 metrics.

In [ ]:
from pathlib import Path
import json
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler

INPUT_PATH = Path('creditcard.csv')
MODEL_OUTPUT = 'random_forest_model'
METRICS_OUTPUT = Path('artifacts/metrics.json')
SEED = 42
FEATURE_COLUMNS = ['Time', *[f'V{i}' for i in range(1, 29)], 'Amount']
spark = SparkSession.builder.appName('LedgerMindFraudTraining').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')

## 1. Read and inspect the dataset

In Colab, upload `creditcard.csv` or replace `INPUT_PATH` with an HDFS-compatible URI. A local Colab path is a simulation, not a real HDFS cluster.

In [ ]:
data = spark.read.option('header', True).option('inferSchema', True).csv(str(INPUT_PATH))
required = [*FEATURE_COLUMNS, 'Class']
missing = sorted(set(required) - set(data.columns))
assert not missing, f'Missing columns: {missing}'
data.select('Class').groupBy('Class').count().orderBy('Class').show()
data.printSchema()

## 2. Clean data and handle class imbalance

The fraud class is very small. Instead of collecting or duplicating the full dataset in Python, Spark class weights are used during Random Forest training.

In [ ]:
data = data.select(*[F.col(c).cast('double').alias(c) for c in required]).na.drop()
normal_count = data.filter(F.col('Class') == 0.0).count()
fraud_count = data.filter(F.col('Class') == 1.0).count()
assert normal_count > 0 and fraud_count > 0
fraud_weight = normal_count / fraud_count
weighted = data.withColumn('classWeight', F.when(F.col('Class') == 1.0, F.lit(fraud_weight)).otherwise(F.lit(1.0)))
train, test = weighted.randomSplit([0.8, 0.2], seed=SEED)
print({'normalCount': normal_count, 'fraudCount': fraud_count, 'fraudWeight': fraud_weight})

## 3. Train Spark ML Random Forest

In [ ]:
assembler = VectorAssembler(inputCols=FEATURE_COLUMNS, outputCol='features')
train_features = assembler.transform(train)
test_features = assembler.transform(test)
classifier = RandomForestClassifier(labelCol='Class', featuresCol='features', weightCol='classWeight', numTrees=100, maxDepth=5, seed=SEED)
model = classifier.fit(train_features)
predictions = model.transform(test_features).cache()
model.write().overwrite().save(MODEL_OUTPUT)

## 4. Evaluate the fraud class

Recall is especially important for fraud detection, but Precision and F1-score must also be reported to avoid flagging every transaction.

In [ ]:
counts = predictions.agg(
    F.sum(F.when((F.col('Class') == 1.0) & (F.col('prediction') == 1.0), 1).otherwise(0)).alias('tp'),
    F.sum(F.when((F.col('Class') == 0.0) & (F.col('prediction') == 1.0), 1).otherwise(0)).alias('fp'),
    F.sum(F.when((F.col('Class') == 1.0) & (F.col('prediction') == 0.0), 1).otherwise(0)).alias('fn'),
    F.count(F.lit(1)).alias('testCount')
).first()
tp, fp, fn = [int(counts[c] or 0) for c in ['tp', 'fp', 'fn']]
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
pr_auc = BinaryClassificationEvaluator(labelCol='Class', rawPredictionCol='rawPrediction', metricName='areaUnderPR').evaluate(predictions)
metrics = {'precision': precision, 'recall': recall, 'f1': f1, 'areaUnderPR': pr_auc, 'truePositive': tp, 'falsePositive': fp, 'falseNegative': fn, 'featureColumns': FEATURE_COLUMNS, 'seed': SEED}
print(json.dumps(metrics, indent=2, default=float))
METRICS_OUTPUT.parent.mkdir(exist_ok=True)
METRICS_OUTPUT.write_text(json.dumps(metrics, indent=2, default=float), encoding='utf-8')

## 5. Reproducibility and shutdown

Record the dataset source/hash, Spark version, parameters, feature order, and metrics in the report. Stop the Spark session after the notebook run.

In [ ]:
print('Spark version:', spark.version)
predictions.unpersist()
spark.stop()